In [3]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

ModuleNotFoundError: No module named 'xgboost'

In [3]:
x_train = pd.read_csv("../Dataset/x_train.csv")
x_test = pd.read_csv("../Dataset/x_test.csv")
y_train = pd.read_csv("../Dataset/y_train.csv").squeeze()
y_test = pd.read_csv("../Dataset/y_test.csv").squeeze()

In [4]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

In [5]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [6]:
search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    random_state=42,
    verbose=2,
    n_jobs=-1
)

In [10]:
print(x_train.shape)
print(y_train.shape)

(8259, 157)
(8259,)


In [11]:
print(x_train.dtypes)

Unit of Measure (Per Pack)                          int64
Line Item Quantity                                  int64
Line Item Value                                   float64
Pack Price                                        float64
Unit Price                                        float64
                                                   ...   
Manufacturing Site_Other                             bool
Manufacturing Site_Standard Diagnostics, Korea       bool
Manufacturing Site_Strides, Bangalore, India.        bool
Manufacturing Site_Trinity Biotech, Plc              bool
First Line Designation_Yes                           bool
Length: 157, dtype: object


In [12]:
print(y_train.unique())

[0 1]


In [14]:
print(x_train.isnull().sum().sum())

0


In [16]:
try:
    search.fit(x_train, y_train)
except Exception as e:
    print(type(e).__name__)
    print(e)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ValueError

All the 100 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\CHENREDDY BHAVYA\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\CHENREDDY BHAVYA\anaconda3\Lib\site-packages\xgboost\core.py", line 553, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\CHENREDDY BHAVYA\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1788, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\CHENREDDY BHAVYA\anacond

In [17]:
print(x_train.dtypes)

Unit of Measure (Per Pack)                          int64
Line Item Quantity                                  int64
Line Item Value                                   float64
Pack Price                                        float64
Unit Price                                        float64
                                                   ...   
Manufacturing Site_Other                             bool
Manufacturing Site_Standard Diagnostics, Korea       bool
Manufacturing Site_Strides, Bangalore, India.        bool
Manufacturing Site_Trinity Biotech, Plc              bool
First Line Designation_Yes                           bool
Length: 157, dtype: object


In [18]:
x_train.columns = (
    x_train.columns.astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
)

x_test.columns = (
    x_test.columns.astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
)

In [19]:
for col in x_train.columns:
    if any(ch in str(col) for ch in ["[", "]", "<"]):
        print(col)

In [20]:
search.fit(x_train,y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_const...
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.8, 1.0],
                                        'learning_rate': [0.01, 0.1, 0.2],
                                        'max_depth': [3, 5, 7],
                                        'n_estimators': [100, 200, 300],
                                        'subsample': [0.8, 1.0]},
                   random_state=42, scoring='accuracy', verbose=2)

In [22]:
print(search.best_params_)

{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [23]:
print(search.best_score_)

0.8961139554426623


In [24]:
best_model = search.best_estimator_

In [25]:
y_pred = best_model.predict(x_test)

In [26]:
import joblib

In [27]:
joblib.dump(best_model, "../Models/xgboost_best_model.pkl")

['../Models/xgboost_best_model.pkl']

In [30]:
import os

print(os.listdir("../Models"))

['xgboost_best_model.pkl']


In [1]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("Tuned XGBoost Performance")
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred))

Tuned XGBoost Performance


NameError: name 'y_test' is not defined